# 02 — Querying the REF API

This notebook shows how to talk to the public REF API: 
set up the client, discover diagnostics, retrieve metric values, and inspect an execution and its output files.

**Prerequisites:** [01 — REF concepts](01-ref-concepts.ipynb).

**What you need:** an internet connection.

## The API and its client

The REF API is documented with an [OpenAPI](https://www.openapis.org) schema.
You can browse the interactive docs at <https://api.climate-ref.org/docs>.

The `ref_tutorials` helper from `climate_ref_client` package builds a client for us:

In [2]:
from ref_tutorials import get_client

client = get_client()

Each API endpoint is a function under `climate_ref_client.api`. 
The functions take a `client=` argument and have a `.sync(...)` method that performs the request and returns a parsed response. 

The payload is on the `.data` attribute.

## Listing diagnostics

In [3]:
from climate_ref_client.api.diagnostics import diagnostics_list

diagnostics = diagnostics_list.sync(client=client).data
len(diagnostics)

47

In [4]:
import pandas as pd

pd.DataFrame(
    {"provider": d.provider.slug, "slug": d.slug, "name": d.name}
    for d in sorted(diagnostics, key=lambda d: (d.provider.slug, d.slug))
)

,provider,slug,name
0,esmvaltool,climate-at-global-warming-levels,Climate at Global Warming Levels
1,esmvaltool,climate-drivers-for-fire,Climate Drivers for Fire
2,esmvaltool,cloud-radiative-effects,Cloud Radiative Effects
3,esmvaltool,cloud-scatterplots-cli-ta,Cloud-Temperature Scatterplots (cli vs ta)
4,esmvaltool,cloud-scatterplots-clivi-lwcre,Cloud-Radiation Scatterplots (clivi vs lwcre)
5,esmvaltool,cloud-scatterplots-clt-swcre,Cloud-Radiation Scatterplots (clt vs swcre)
6,esmvaltool,cloud-scatterplots-clwvi-pr,Cloud-Precipitation Scatterplots (clwvi vs pr)
7,esmvaltool,cloud-scatterplots-reference,Cloud Scatterplots for Reference dataset
8,esmvaltool,enso-basic-climatology,ENSO Basic Climatology
9,esmvaltool,enso-characteristics,ENSO Characteristics


## Retrieving metric values

Many diagnostics expose **scalar** metric values. 

We pick one diagnostic and ask the API for its scalar values.
Each value carries a set of *dimensions* (model, statistic, region, ...) and a single number.

In [5]:
from climate_ref_client.api.diagnostics import diagnostics_list_metric_values
from climate_ref_client.models.metric_value_type import MetricValueType

result = diagnostics_list_metric_values.sync(
    provider_slug="esmvaltool",
    diagnostic_slug="climate-at-global-warming-levels",
    value_type=MetricValueType.SCALAR,
    client=client,
)
values = result.data
print(f"Climate at Global Warming Levels: {len(values)} scalar metric values")
values[0]

Climate at Global Warming Levels: 44 scalar metric values


ScalarValue(dimensions=ScalarValueDimensions(additional_properties={'experiment_id': 'ssp126', 'metric': 'exceedance_year'}), value=2028.0, id=1, execution_group_id=1, execution_id=1, attributes=None, is_outlier=False, verification_status=<ScalarValueVerificationStatusType0.VERIFIED: 'verified'>, additional_properties={})

Raw metric values are awkward to work with. The `ref_tutorials` helper flattens them
into a tidy `pandas.DataFrame` — one column per dimension, plus a `value` column:# ~mikapfl: I don't understand *what* is exceeded in the exceedance year - how would I find out?

In [6]:
from ref_tutorials import metric_values_to_dataframe

df = metric_values_to_dataframe(values)
df.head(10)

,experiment_id,metric,value
0,ssp126,exceedance_year,2028.0
1,ssp126,exceedance_year,2040.0
2,ssp245,exceedance_year,2031.0
3,ssp245,exceedance_year,2044.0
4,ssp245,exceedance_year,2069.0
5,ssp245,exceedance_year,2081.0
6,ssp585,exceedance_year,2029.0
7,ssp585,exceedance_year,2036.0
8,ssp585,exceedance_year,2052.0
9,ssp585,exceedance_year,2068.0


From here it is ordinary pandas — filter, group, and summarise as you like:

In [7]:
df["value"].describe()

count      21.000000
mean     2045.952381
std        19.868257
min      2020.000000
25%      2031.000000
50%      2040.000000
75%      2060.000000
max      2082.000000
Name: value, dtype: float64

## Inspecting an execution

Beyond metric values, executions produce **output files** (NetCDF data, figures). We
fetch one execution group and list what it produced.

In [8]:
from climate_ref_client.api.executions import executions_get

execution = executions_get.sync(diagnostic.execution_groups[0], client=client)
print("execution key:", execution.key)

outputs = execution.latest_execution.outputs
for output in outputs[:10]:
    print("  ", output.filename)

execution key: cmip6_ssp126
   executions/recipe_20260305_234552/plots/gwl_mean_plots_pr/plot_gwl_stats/CMIP6_mm_mean_1.5.png
   executions/recipe_20260305_234552/plots/gwl_mean_plots_pr/plot_gwl_stats/CMIP6_mm_mean_2.0.png
   executions/recipe_20260305_234552/plots/gwl_mean_plots_pr/plot_gwl_stats/CMIP6_mm_stdev_1.5.png
   executions/recipe_20260305_234552/plots/gwl_mean_plots_pr/plot_gwl_stats/CMIP6_mm_stdev_2.0.png
   executions/recipe_20260305_234552/plots/gwl_mean_plots_tas/plot_gwl_stats/CMIP6_mm_mean_1.5.png
   executions/recipe_20260305_234552/plots/gwl_mean_plots_tas/plot_gwl_stats/CMIP6_mm_mean_2.0.png
   executions/recipe_20260305_234552/plots/gwl_mean_plots_tas/plot_gwl_stats/CMIP6_mm_stdev_1.5.png
   executions/recipe_20260305_234552/plots/gwl_mean_plots_tas/plot_gwl_stats/CMIP6_mm_stdev_2.0.png
   executions/recipe_20260305_234552/work/calculate_gwl_exceedance_years/gwl_exceedance_calculation/GWL_exceedance_years.csv
   executions/recipe_20260305_234552/work/gwl_mean_plot

Each output has a `url`. You can download it with `requests` and open NetCDF files with
`xarray` for your own analysis — that is exactly what notebook 03 does to build a figure.

## Recap

- Build a client with `get_client()`.
- API endpoints live under `climate_ref_client.api`; call `.sync(client=client)` and
  read `.data`.
- `diagnostics_list` discovers diagnostics; `diagnostics_list_metric_values` fetches
  their numbers; `executions_get` inspects a single run.
- `metric_values_to_dataframe` turns metric values into tidy pandas.

**Next:** [03 — A publication-ready figure](03-publication-figure.ipynb).